In [ ]:
import os
import sys
import random
import statistics
from pathlib import Path

import numpy as np
import open3d as o3d
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "table3_results":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == "Replicability":
    PROJECT_ROOT = PROJECT_ROOT.parent

REPLICABILITY_DIR = PROJECT_ROOT / "Replicability"
BASELINES_CHECKPOINTS_DIR = PROJECT_ROOT / "Baselines_Checkpoints"
DATASETS_DIR = REPLICABILITY_DIR / "datasets"
LION_CHECKPOINTS_DIR = BASELINES_CHECKPOINTS_DIR / "LION"
LION_ORIGINAL_CHECKPOINTS_DIR = LION_CHECKPOINTS_DIR / "original"
LION_MIRRORED_CHECKPOINTS_DIR = LION_CHECKPOINTS_DIR / "mirrored"
ORIGINAL_SHAPENET_DIR = DATASETS_DIR / "ShapeNetCore.v2.PC15k"
MIRRORED_SHAPENET_DIR = DATASETS_DIR / "Mirrored_ShapeNetCore.v2.PC15k"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

SYMMETRY_PROTOCOL_DIR = PROJECT_ROOT / "Symmetry_Measurement_Protocol"
if str(SYMMETRY_PROTOCOL_DIR) not in sys.path:
    sys.path.append(str(SYMMETRY_PROTOCOL_DIR))

from models.vae_adain import Model as VAE
from default_config import cfg
from Householder_transform import householder_transformation

cfg_original_airplane = cfg.clone()
cfg_mirrored_airplane = cfg.clone()

cfg_original_car = cfg.clone()
cfg_mirrored_car = cfg.clone()

cfg_original_chair = cfg.clone()
cfg_mirrored_chair = cfg.clone()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# Path to ckpt
## Airplane
ckpt_path_original_airplane = LION_ORIGINAL_CHECKPOINTS_DIR / "vae_only_airplane.pt"
cfg_path_original_airplane = LION_ORIGINAL_CHECKPOINTS_DIR / "cfg_airplane.yml"

ckpt_path_mirrored_airplane = LION_MIRRORED_CHECKPOINTS_DIR / "vae_airplane_epoch_7999_iters_175999.pt"
cfg_path_mirrored_airplane = LION_MIRRORED_CHECKPOINTS_DIR / "cfg_airplane.yml"

input_original_dir_airplane = ORIGINAL_SHAPENET_DIR / "02691156" / "val"
input_mirrored_dir_airplane = MIRRORED_SHAPENET_DIR / "02691156" / "val"

cfg_original_airplane.merge_from_file(str(cfg_path_original_airplane))
args_original_airplane = cfg_original_airplane

cfg_mirrored_airplane.merge_from_file(str(cfg_path_mirrored_airplane))
args_mirrored_airplane = cfg_mirrored_airplane

# Load ckpt
ckpt_original_airplane = torch.load(ckpt_path_original_airplane, map_location=device)
ckpt_mirrored_airplane = torch.load(ckpt_path_mirrored_airplane, map_location=device)

# Instanciate model
vae_original_airplane = VAE(args_original_airplane).to(device)
vae_original_airplane.load_state_dict(ckpt_original_airplane["model"])
vae_original_airplane.eval()

vae_mirrored_airplane = VAE(args_mirrored_airplane).to(device)
vae_mirrored_airplane.load_state_dict(ckpt_mirrored_airplane["model"])
vae_mirrored_airplane.eval()

## Car
ckpt_path_original_car = LION_ORIGINAL_CHECKPOINTS_DIR / "vae_only_car.pt"
cfg_path_original_car = LION_ORIGINAL_CHECKPOINTS_DIR / "cfg_car.yml"

ckpt_path_mirrored_car = LION_MIRRORED_CHECKPOINTS_DIR / "vae_car_epoch_7999_iters_151999.pt"
cfg_path_mirrored_car = LION_MIRRORED_CHECKPOINTS_DIR / "cfg_car.yml"

input_original_dir_car = ORIGINAL_SHAPENET_DIR / "02958343" / "val"
input_mirrored_dir_car = MIRRORED_SHAPENET_DIR / "02958343" / "val"

cfg_original_car.merge_from_file(str(cfg_path_original_car))
args_original_car = cfg_original_car

cfg_mirrored_car.merge_from_file(str(cfg_path_mirrored_car))
args_mirrored_car = cfg_mirrored_car

# Load ckpt
ckpt_original_car = torch.load(ckpt_path_original_car, map_location=device)
ckpt_mirrored_car = torch.load(ckpt_path_mirrored_car, map_location=device)

# Instanciate model
vae_original_car = VAE(args_original_car).to(device)
vae_original_car.load_state_dict(ckpt_original_car["model"])
vae_original_car.eval()

vae_mirrored_car = VAE(args_mirrored_car).to(device)
vae_mirrored_car.load_state_dict(ckpt_mirrored_car["model"])
vae_mirrored_car.eval()

## Chair
ckpt_path_original_chair = LION_ORIGINAL_CHECKPOINTS_DIR / "vae_only_chair.pt"
cfg_path_original_chair = LION_ORIGINAL_CHECKPOINTS_DIR / "cfg_chair.yml"

ckpt_path_mirrored_chair = LION_MIRRORED_CHECKPOINTS_DIR / "vae_chair_epoch_7999_iters_287999.pt"
cfg_path_mirrored_chair = LION_MIRRORED_CHECKPOINTS_DIR / "cfg_chair.yml"

input_original_dir_chair = ORIGINAL_SHAPENET_DIR / "03001627" / "val"
input_mirrored_dir_chair = MIRRORED_SHAPENET_DIR / "03001627" / "val"

cfg_original_chair.merge_from_file(str(cfg_path_original_chair))
args_original_chair = cfg_original_chair

cfg_mirrored_chair.merge_from_file(str(cfg_path_mirrored_chair))
args_mirrored_chair = cfg_mirrored_chair

# Load ckpt
ckpt_original_chair = torch.load(ckpt_path_original_chair, map_location=device)
ckpt_mirrored_chair = torch.load(ckpt_path_mirrored_chair, map_location=device)

# Instanciate model
vae_original_chair = VAE(args_original_chair).to(device)
vae_original_chair.load_state_dict(ckpt_original_chair["model"])
vae_original_chair.eval()

vae_mirrored_chair = VAE(args_mirrored_chair).to(device)
vae_mirrored_chair.load_state_dict(ckpt_mirrored_chair["model"])
vae_mirrored_chair.eval()


### Helpers

In [3]:
# Mean and std for original objects
## Airplane
original_airplane_mean = torch.tensor([0.000201, 0.006636, 0.058967], device=device).view(1, 1, 3) # [1, 1, 3]
original_airplane_std = torch.tensor([0.117082], device=device).view(1, 1, 1) # [1, 1, 1]
## Car
original_car_mean = torch.tensor([0.001059, 0.008262, 0.019432], device=device).view(1, 1, 3) # [1, 1, 3]
original_car_std = torch.tensor([0.163453], device=device).view(1, 1, 1) # [1, 1, 1]
## Chair
original_chair_mean = torch.tensor([0.000542, 0.000341, 0.000993], device=device).view(1, 1, 3) # [1, 1, 3]
original_chair_std = torch.tensor([0.170389], device=device).view(1, 1, 1) # [1, 1, 1]

# Mean and std for mirrored objects
## Airplane
mirrored_airplane_mean = torch.tensor([0.000001, 0.006686, 0.056193], device=device).view(1, 1, 3) # [1, 1, 3]
mirrored_airplane_std = torch.tensor([0.116419], device=device).view(1, 1, 1) # [1, 1, 1]
## Car
mirrored_car_mean = torch.tensor([0.000007, 0.008659, 0.020091], device=device).view(1, 1, 3) # [1, 1, 3]
mirrored_car_std = torch.tensor([0.162546], device=device).view(1, 1, 1) # [1, 1, 1]
## Chair
mirrored_chair_mean = torch.tensor([-0.000022, 0.000207, 0.000578], device=device).view(1, 1, 3) # [1, 1, 3]
mirrored_chair_std = torch.tensor([0.171713], device=device).view(1, 1, 1) # [1, 1, 1]

In [ ]:
def compute_average_l2_from_pairs(files, directory, vae, g_class):
    l2 = []

    for a in files:
        file_path_A = os.path.join(directory, a)

        # Take another random pc
        b = random.choice(files)

        if b == a:
            b = random.choice(files)
        
        file_path_B = os.path.join(directory, b)

        # Load pcs
        pc_A = np.load(file_path_A)
        pc_B = np.load(file_path_B)

        if pc_A.ndim == 2 and pc_B.ndim == 2: # Matriz
            pc_A = pc_A[None, ...]
            pc_B = pc_B[None, ...]

        A_tensor = torch.tensor(pc_A, dtype=torch.float32).to(device)
        B_tensor = torch.tensor(pc_B, dtype=torch.float32).to(device)

        # Get latents (mu)
        if vae == "vae_original":
            if g_class == "airplane":
                A_tensor = (A_tensor - original_airplane_mean) / original_airplane_std
                B_tensor = (B_tensor - original_airplane_mean) / original_airplane_std

                mu_A = vae_original_airplane.get_latents_mu(A_tensor)
                mu_B = vae_original_airplane.get_latents_mu(B_tensor)
            elif g_class == "car":
                A_tensor = (A_tensor - original_car_mean) / original_car_std
                B_tensor = (B_tensor - original_car_mean) / original_car_std

                mu_A = vae_original_car.get_latents_mu(A_tensor)
                mu_B = vae_original_car.get_latents_mu(B_tensor)
            elif g_class == "chair":
                A_tensor = (A_tensor - original_chair_mean) / original_chair_std
                B_tensor = (B_tensor - original_chair_mean) / original_chair_std

                mu_A = vae_original_chair.get_latents_mu(A_tensor)
                mu_B = vae_original_chair.get_latents_mu(B_tensor)

        elif vae == "vae_mirrored":
            if g_class == "airplane":
                A_tensor = (A_tensor - mirrored_airplane_mean) / mirrored_airplane_std
                B_tensor = (B_tensor - mirrored_airplane_mean) / mirrored_airplane_std

                mu_A = vae_mirrored_airplane.get_latents_mu(A_tensor)
                mu_B = vae_mirrored_airplane.get_latents_mu(B_tensor)
            elif g_class == "car":
                A_tensor = (A_tensor - mirrored_car_mean) / mirrored_car_std
                B_tensor = (B_tensor - mirrored_car_mean) / mirrored_car_std

                mu_A = vae_mirrored_car.get_latents_mu(A_tensor)
                mu_B = vae_mirrored_car.get_latents_mu(B_tensor)
            elif g_class == "chair":
                A_tensor = (A_tensor - mirrored_chair_mean) / mirrored_chair_std
                B_tensor = (B_tensor - mirrored_chair_mean) / mirrored_chair_std

                mu_A = vae_mirrored_chair.get_latents_mu(A_tensor)
                mu_B = vae_mirrored_chair.get_latents_mu(B_tensor)

        # Compute l2
        computed_l2 = torch.norm(mu_A[0] - mu_B[0], dim=-1)

        l2.append(computed_l2.item())

    return statistics.mean(l2)

def compute_l2_list_from_original_mirrored(files, directory, average_l2_from_pairs, vae, g_class):
    means = []
    cont = 0

    for x in files:
        file_path_pc = os.path.join(directory, x)
        original_pc = np.load(file_path_pc)

        if original_pc.shape[0] == 15000:
            pass
        elif original_pc.shape[0] == 30000:
            #pass
            original_pc = farthest_point_sampling(original_pc, 15000)
        
        # Obtain the mirrored version
        mirrored_pc = householder_transformation(original_pc)

        # Convert to tensors
        if original_pc.ndim == 2 and mirrored_pc.ndim == 2:
            original_pc = original_pc[None, ...]
            mirrored_pc = mirrored_pc[None, ...]
        
        original_pc = torch.tensor(original_pc, dtype=torch.float32).to(device)
        mirrored_pc = torch.tensor(mirrored_pc, dtype=torch.float32).to(device)
    
        if vae == "vae_original":
            if g_class == "airplane":
                original_pc = (original_pc - original_airplane_mean) / original_airplane_std
                mirrored_pc = (mirrored_pc - original_airplane_mean) / original_airplane_std

                mu_original = vae_original_airplane.get_latents_mu(original_pc)
                mu_mirrored = vae_original_airplane.get_latents_mu(mirrored_pc)
            elif g_class == "car":
                original_pc = (original_pc - original_car_mean) / original_car_std
                mirrored_pc = (mirrored_pc - original_car_mean) / original_car_std

                mu_original = vae_original_car.get_latents_mu(original_pc)
                mu_mirrored = vae_original_car.get_latents_mu(mirrored_pc)
            elif g_class == "chair":
                original_pc = (original_pc - original_chair_mean) / original_chair_std
                mirrored_pc = (mirrored_pc - original_chair_mean) / original_chair_std

                mu_original = vae_original_chair.get_latents_mu(original_pc)
                mu_mirrored = vae_original_chair.get_latents_mu(mirrored_pc)

        elif vae == "vae_mirrored":
            if g_class == "airplane":
                original_pc = (original_pc - mirrored_airplane_mean) / mirrored_airplane_std
                mirrored_pc = (mirrored_pc - mirrored_airplane_mean) / mirrored_airplane_std

                mu_original = vae_mirrored_airplane.get_latents_mu(original_pc)
                mu_mirrored = vae_mirrored_airplane.get_latents_mu(mirrored_pc)
            elif g_class == "car":
                original_pc = (original_pc - mirrored_car_mean) / mirrored_car_std
                mirrored_pc = (mirrored_pc - mirrored_car_mean) / mirrored_car_std

                mu_original = vae_mirrored_car.get_latents_mu(original_pc)
                mu_mirrored = vae_mirrored_car.get_latents_mu(mirrored_pc)
            elif g_class == "chair":
                original_pc = (original_pc - mirrored_chair_mean) / mirrored_chair_std
                mirrored_pc = (mirrored_pc - mirrored_chair_mean) / mirrored_chair_std

                mu_original = vae_mirrored_chair.get_latents_mu(original_pc)
                mu_mirrored = vae_mirrored_chair.get_latents_mu(mirrored_pc)

        numerator = torch.norm(mu_original[0] - mu_mirrored[0], dim=-1)
        denominator = average_l2_from_pairs

        result = numerator / denominator

        means.append(result.item())

        cont += 1

    return means

def farthest_point_sampling(points, n_samples=2048):
    # Create a point cloud of Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    # compute FPS
    downpcd_farthest = pcd.farthest_point_down_sample(n_samples)

    return np.asarray(downpcd_farthest.points)

## Latents from original encoder

### Airplane

In [5]:
original_files_airplane = sorted([f for f in os.listdir(input_original_dir_airplane) if f.endswith(".npy")])#[:5]

average_l2_from_pairs_airplane = compute_average_l2_from_pairs(original_files_airplane[:100], input_original_dir_airplane, "vae_original", "airplane")
l2_list_from_original_mirrored_airplane = compute_l2_list_from_original_mirrored(original_files_airplane, input_original_dir_airplane, average_l2_from_pairs_airplane, "vae_original", "airplane")

print(l2_list_from_original_mirrored_airplane)
print("\nAverage mean " + str(len(original_files_airplane)) +":", statistics.mean(l2_list_from_original_mirrored_airplane))
print("Average mean/max:", max(l2_list_from_original_mirrored_airplane))
print("Average mean/min:", min(l2_list_from_original_mirrored_airplane))

[0.1432885378599167, 0.21068665385246277, 0.03510376438498497, 0.0992916077375412, 0.14435680210590363, 0.15572801232337952, 0.09116265177726746, 0.12092214077711105, 0.09525246918201447, 0.21718627214431763, 0.07954086363315582, 0.16438020765781403, 0.09335672855377197, 0.059571556746959686, 0.045137643814086914, 0.2650967240333557, 0.10963684320449829, 0.07389841228723526, 0.0785495713353157, 0.09667479991912842, 0.153585746884346, 0.13657279312610626, 0.19081264734268188, 0.10817864537239075, 0.21154430508613586, 0.15496885776519775, 0.13776108622550964, 0.21189916133880615, 0.08490407466888428, 0.06144491210579872, 0.08978331834077835, 0.08870904892683029, 0.15038006007671356, 0.08194224536418915, 0.0940614640712738, 0.20277543365955353, 0.07503946870565414, 0.08195486664772034, 0.10920727252960205, 0.06528498232364655, 0.08895279467105865, 0.15924587845802307, 0.2878267467021942, 0.08642871677875519, 0.11216438561677933, 0.3476185202598572, 0.10969779640436172, 0.2770397365093231,

### Car

In [6]:
original_files_car = sorted([f for f in os.listdir(input_original_dir_car) if f.endswith(".npy")])#[:5]

average_l2_from_pairs_car = compute_average_l2_from_pairs(original_files_car[:100], input_original_dir_car, "vae_original", "car")
l2_list_from_original_mirrored_car = compute_l2_list_from_original_mirrored(original_files_car, input_original_dir_car, average_l2_from_pairs_car, "vae_original", "car")

print(l2_list_from_original_mirrored_car)
print("\nAverage mean " + str(len(original_files_car)) +":", statistics.mean(l2_list_from_original_mirrored_car))
print("Average mean/max:", max(l2_list_from_original_mirrored_car))
print("Average mean/min:", min(l2_list_from_original_mirrored_car))

[0.05097636207938194, 0.07879015058279037, 0.054043397307395935, 0.14380218088626862, 0.06647349148988724, 0.038642607629299164, 0.07013730704784393, 0.05477644503116608, 0.08417654782533646, 0.0791700929403305, 0.19771046936511993, 0.03569776937365532, 0.11312123388051987, 0.026493525132536888, 0.07859019935131073, 0.0710446760058403, 0.042471326887607574, 0.04767507687211037, 0.15280993282794952, 0.1363459676504135, 0.06613302230834961, 0.058342546224594116, 0.05482608079910278, 0.02277395687997341, 0.015728959813714027, 0.07507748156785965, 0.12356368452310562, 0.032114725559949875, 0.17597617208957672, 0.09656141698360443, 0.05702551081776619, 0.04335862025618553, 0.09644537419080734, 0.04894891381263733, 0.26479142904281616, 0.064580999314785, 0.030484236776828766, 0.06867419928312302, 0.08281836658716202, 0.09368567913770676, 0.060532864183187485, 0.04162572696805, 0.030764946714043617, 0.04623771086335182, 0.08116432279348373, 0.04227852076292038, 0.025076355785131454, 0.0623495

### Chair

In [7]:
original_files_chair = sorted([f for f in os.listdir(input_original_dir_chair) if f.endswith(".npy")])#[:5]

average_l2_from_pairs_chair = compute_average_l2_from_pairs(original_files_chair[:100], input_original_dir_chair, "vae_original", "chair")
l2_list_from_original_mirrored_chair = compute_l2_list_from_original_mirrored(original_files_chair, input_original_dir_chair, average_l2_from_pairs_chair, "vae_original", "chair")

print(l2_list_from_original_mirrored_chair)
print("\nAverage mean " + str(len(original_files_chair)) +":", statistics.mean(l2_list_from_original_mirrored_chair))
print("Average mean/max:", max(l2_list_from_original_mirrored_chair))
print("Average mean/min:", min(l2_list_from_original_mirrored_chair))

[0.06107347458600998, 0.1358109414577484, 0.13459032773971558, 0.040966182947158813, 0.0722639337182045, 0.06325868517160416, 0.09988480806350708, 0.08656181395053864, 0.058460675179958344, 0.08921003341674805, 0.09626924246549606, 0.08954209834337234, 0.05138993263244629, 0.11664030700922012, 0.0660744458436966, 0.0923074260354042, 0.16912560164928436, 0.03724838048219681, 0.14236529171466827, 0.07610086351633072, 0.2694442868232727, 0.05956575646996498, 0.06486334651708603, 0.0834067091345787, 0.07999946922063828, 0.04686116799712181, 0.108443982899189, 0.04541206359863281, 0.04650139808654785, 0.08460685610771179, 0.07899324595928192, 0.14776133000850677, 0.3095487058162689, 0.05024796724319458, 0.1016276478767395, 0.30397137999534607, 0.04548754170536995, 0.30053168535232544, 0.07502586394548416, 0.049666620790958405, 0.12233144044876099, 0.08665130287408829, 0.04754723235964775, 0.03996305540204048, 0.10345321893692017, 0.05773460865020752, 0.034314628690481186, 0.0405755043029785

## Latents from mirrored encoder

### Airplane

In [8]:
mirrored_files_airplane = sorted([f for f in os.listdir(input_mirrored_dir_airplane) if f.endswith(".npy")])#[:5]

average_l2_from_pairs_airplane = compute_average_l2_from_pairs(mirrored_files_airplane[:100], input_mirrored_dir_airplane, "vae_mirrored", "airplane")
l2_list_from_original_mirrored_airplane = compute_l2_list_from_original_mirrored(mirrored_files_airplane, input_mirrored_dir_airplane, average_l2_from_pairs_airplane, "vae_mirrored", "airplane")

print(l2_list_from_original_mirrored_airplane)
print("\nAverage mean " + str(len(mirrored_files_airplane)) +":", statistics.mean(l2_list_from_original_mirrored_airplane))
print("Average mean/max:", max(l2_list_from_original_mirrored_airplane))
print("Average mean/min:", min(l2_list_from_original_mirrored_airplane))

[0.06285474449396133, 0.09085893630981445, 0.055688366293907166, 0.08311457186937332, 0.06496668606996536, 0.040306877344846725, 0.04489591717720032, 0.05490521341562271, 0.06749767810106277, 0.07116319239139557, 0.04941188171505928, 0.08198382705450058, 0.08483340591192245, 0.054534927010536194, 0.06515326350927353, 0.052974358201026917, 0.03932415693998337, 0.07956160604953766, 0.06397934257984161, 0.02714286744594574, 0.06429877132177353, 0.08409485965967178, 0.09816937148571014, 0.10427214205265045, 0.07543572038412094, 0.08369904011487961, 0.05474632978439331, 0.06572312116622925, 0.09092308580875397, 0.04517517238855362, 0.04895253852009773, 0.06322790682315826, 0.08899939060211182, 0.07997318357229233, 0.0842735543847084, 0.0783359706401825, 0.045234810560941696, 0.08315859735012054, 0.07416002452373505, 0.05399655923247337, 0.06640657037496567, 0.07617134600877762, 0.07106628268957138, 0.10180212557315826, 0.08374790847301483, 0.12426561117172241, 0.04527629166841507, 0.0823654

### Car

In [9]:
mirrored_files_car = sorted([f for f in os.listdir(input_mirrored_dir_car) if f.endswith(".npy")])#[:5]

average_l2_from_pairs_car = compute_average_l2_from_pairs(mirrored_files_car[:100], input_mirrored_dir_car, "vae_mirrored", "car")
l2_list_from_original_mirrored_car = compute_l2_list_from_original_mirrored(mirrored_files_car, input_mirrored_dir_car, average_l2_from_pairs_car, "vae_mirrored", "car")

print(l2_list_from_original_mirrored_car)
print("\nAverage mean " + str(len(mirrored_files_car)) +":", statistics.mean(l2_list_from_original_mirrored_car))
print("Average mean/max:", max(l2_list_from_original_mirrored_car))
print("Average mean/min:", min(l2_list_from_original_mirrored_car))

[0.05039387568831444, 0.03977835178375244, 0.06751397252082825, 0.07295084744691849, 0.03355413302779198, 0.039750006049871445, 0.05191073566675186, 0.03816933184862137, 0.06977564841508865, 0.05081791803240776, 0.07812347263097763, 0.03729591518640518, 0.027015944942831993, 0.047338031232357025, 0.03213372081518173, 0.02962757647037506, 0.028760775923728943, 0.05181180685758591, 0.03740444779396057, 0.07195749133825302, 0.06402059644460678, 0.043305762112140656, 0.04906867817044258, 0.04201468452811241, 0.05334528908133507, 0.033760663121938705, 0.04477817565202713, 0.05796146020293236, 0.04241669550538063, 0.030542058870196342, 0.05989217013120651, 0.049048155546188354, 0.047127172350883484, 0.059446580708026886, 0.04766858369112015, 0.05262744054198265, 0.033939652144908905, 0.03442373499274254, 0.07741132378578186, 0.08617237210273743, 0.055364929139614105, 0.0810932070016861, 0.04085969552397728, 0.022167446091771126, 0.05653560161590576, 0.038629427552223206, 0.0662316381931305, 

### Chair

In [10]:
mirrored_files_chair = sorted([f for f in os.listdir(input_mirrored_dir_chair) if f.endswith(".npy")])#[:5]

average_l2_from_pairs_chair = compute_average_l2_from_pairs(mirrored_files_chair[:100], input_mirrored_dir_chair, "vae_mirrored", "chair")
l2_list_from_original_mirrored_chair = compute_l2_list_from_original_mirrored(mirrored_files_chair, input_mirrored_dir_chair, average_l2_from_pairs_chair, "vae_mirrored", "chair")

print(l2_list_from_original_mirrored_chair)
print("\nAverage mean " + str(len(mirrored_files_chair)) +":", statistics.mean(l2_list_from_original_mirrored_chair))
print("Average mean/max:", max(l2_list_from_original_mirrored_chair))
print("Average mean/min:", min(l2_list_from_original_mirrored_chair))

[0.04337761923670769, 0.054628707468509674, 0.03961476683616638, 0.02166690304875374, 0.03662041574716568, 0.05650680139660835, 0.041339192539453506, 0.04652473330497742, 0.037801593542099, 0.05009807273745537, 0.027081800624728203, 0.0415923073887825, 0.037651509046554565, 0.07865668833255768, 0.03735016658902168, 0.026698851957917213, 0.08146320283412933, 0.03082672320306301, 0.03259396180510521, 0.029870757833123207, 0.07933506369590759, 0.019261203706264496, 0.025701701641082764, 0.04428611323237419, 0.059943776577711105, 0.023624908179044724, 0.0801132544875145, 0.024602457880973816, 0.029978277161717415, 0.037857331335544586, 0.08122649043798447, 0.0552801713347435, 0.08519339561462402, 0.03143174201250076, 0.055912163108587265, 0.10453274101018906, 0.049453869462013245, 0.036231156438589096, 0.03238824009895325, 0.043974049389362335, 0.04848447069525719, 0.03878544270992279, 0.03386140614748001, 0.02357475645840168, 0.04408927634358406, 0.04077114909887314, 0.03505142405629158, 